# Annotated CIF tutorial — set up a pocket-conditioned design in a few steps

**Goal:** starting from a protein–ligand complex, build a single model-ready **annotated CIF** that asks RFProteina to design a new protein around the ligand. In a handful of steps we will make an input with:

- flexible **ligand** with some anchored atoms;
- **one pocket residue** fully fixed;
- two pocket residues with just tip atoms coordinate conditioned;
- mark **H-bond** donor/acceptor roles on the ligand's polar atoms;
- write the CIF and emit a small sweep of variants programmatically.

No JSON, no contig strings — just AtomWorks annotations on an atom array. See the [annotated CIF guide](../guides/annotated_cif_inputs.md) for the full annotation reference.

In [ ]:
from pathlib import Path

import os
import numpy as np
from atomworks.io import parse
from atomworks.io.utils.annotator import ensure_annotations
from atomworks.io.utils.io_utils import to_cif_file
from atomworks.io.utils.visualize import view
from atomworks.ml.conditions import C_CRD, C_IDX, C_SEQ
from atomworks.ml.utils.expandable_segment import insert_expandable_segment

from rfproteina.conditions import C_HB, C_IDXGRP, HBondEnum
from rfproteina.utils.parse_presets import ANNOTATED_CIF, RCSB_WITH_METALS

# Point INPUT_CIF at your own protein-ligand complex. Default: a repo fixture.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".project-root").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / ".project-root").exists():          # e.g. run outside the checkout
    _root = os.environ.get("PROJECT_ROOT")
    if not _root:
        raise RuntimeError(
            "Could not find the repository root. Run this notebook from inside the "
            "RFD4-Proteina checkout (it has a `.project-root` marker), or set PROJECT_ROOT "
            "to the checkout path."
        )
    REPO_ROOT = Path(_root)
INPUT_CIF = REPO_ROOT / "data/benchmarks/tip_atom_plinder_training_examples/5eni__1__1.A__1.B.cif"
OUT_DIR = REPO_ROOT / "runs/annotated_cif_tutorial"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parse and subselect to the pocket

Parse the complex, then keep only the **ligand plus the three pocket residues nearest to it**. In this example, we fully fix `TYR 41` and will pin one tip atom each on `VAL 36` and `THR 87`.

In [ ]:
# Pocket residues nearest the ligand (chain, res_id), found offline.
scaffold_residue_ids = [41, 36, 87]
TIPS = {("1.A", 36): "CG1", ("1.A", 87): "OG1"}  # pin one sidechain tip atom each

parse_config = RCSB_WITH_METALS.replace(hydrogen_policy="remove")

aa = parse(
    str(INPUT_CIF),
    config=parse_config,  # see from rfproteina.utils.parse_presets for common configs you might want for parse
)["assemblies"]["1"][0]
ensure_annotations(
    aa, "is_ligand"
)  # check out atomworks.io.utils.annotator and atomworks.ml.utils.annotator for easily addable annotations!

keep_mask = aa.is_ligand | ((aa.chain_id == "1.A") & np.isin(aa.res_id, scaffold_residue_ids))
aa = aa[keep_mask]

print(aa.array_length(), "atoms |", int(aa.is_ligand.sum()), "ligand atoms")

In [ ]:
# Render inline: the ligand (sticks) surrounded by the three pocket residues.
view(aa)

## 2. Add an expandable segment

In [ ]:
aa = insert_expandable_segment(
    aa,
    seg_min=150,
    seg_max=200,
    new_chain=False,
    atom_idx=0,
    insert_before=True,
)

## 3. Apply the pocket conditioning

Three conditions define the motif:

- **Coordinates** (`C_CRD`) — fix the chlorine ligand atoms.
- **Sequence** (`C_SEQ`) — reveal identity for all scaffolded residues and the ligand.
- **Index** (`C_IDX` / `C_IDXGRP`) — mark the three motif residues **unindexed** (their position in the designed sequence is free) and put each in its **own index group** so they place independently.

In [ ]:
coord_cond_mask = (aa.chain_id == "1.A") & (aa.res_id == 41)  # residue 41: all atoms
for (chain, res_id), atom in TIPS.items():                     # plus one pinned tip atom each
    coord_cond_mask |= (aa.chain_id == chain) & (aa.res_id == res_id) & (aa.atom_name == atom)
scaffold_residues = np.isin(aa.res_id, scaffold_residue_ids) & (aa.chain_id == "1.A")

# Coordinates: ligand + fully-fixed residue (all atoms) + the pinned tip atoms.
C_CRD.set_mask(aa, coord_cond_mask)

# Sequence: condition ligand and scaffolded residues (not expandable segment)
C_SEQ.set_mask(aa, scaffold_residues | aa.is_ligand)

# Index: unindex all three motif residues, then ensure each residue has it's own index group.
C_IDX.set_mask(aa, ~scaffold_residues)

# The unindexed residues have non-contiguous residue IDs, so the default IDXGROUP would already
# put each in its own group. We reset it explicitly here anyway, just to show how - this step
# isn't strictly necessary given the default behavior.
groups = C_IDXGRP.annotation(aa, default="generate")
groups[aa.res_id == 36] = max(groups) + 1
groups[aa.res_id == 87] = max(groups) + 1
assert (
    len(np.unique(groups[scaffold_residues])) == 3
), "Expected three unique index groups for the three scaffold residues."
assert (
    len(np.unique(groups[~scaffold_residues])) == 2
), "Expected two unique index groups for the ligand and expandable segment."
C_IDXGRP.set_annotation(aa, groups)  # Again, not strictly necessary here since we aren't changing from default

print("coord-fixed atoms:", int(C_CRD.mask(aa).sum()), "| unindexed atoms:", (~C_IDX.mask(aa)).sum())

## 4. Hydrogen-bond roles on the ligand

Request donor/acceptor roles on the ligand's polar atoms so the designed pocket satisfies them. `5QA` carries an amide plus a hydroxyl: the amide `N` **donates**, the carbonyl `O1` **accepts**, and the hydroxyl `O` is an acceptor (annotating that an atom is `BOTH` a donor and an acceptor is coming soon!).

In [ ]:
hbond = C_HB.default_annotation(aa).copy()
hbond[aa.is_ligand & (aa.atom_name == "N")] = HBondEnum.DONOR  # amide N
hbond[aa.is_ligand & (aa.atom_name == "O1")] = HBondEnum.ACCEPTOR  # carbonyl O
hbond[aa.is_ligand & (aa.atom_name == "O")] = HBondEnum.ACCEPTOR  # hydroxyl O
C_HB.set_annotation(aa, hbond.astype(int))
print("H-bond atoms set:", int(C_HB.mask(aa).sum()))

## 5. Write the annotated CIF

Write with `save_standard_annotations=True` (all conditions embedded), then re-parse with the `ANNOTATED_CIF` preset — the exact path the model uses — to confirm every condition survives the round-trip.

In [ ]:
out_path = OUT_DIR / "annotated_design.cif"
# save_standard_annotations defaults to True; shown explicitly since this is what saves all
# conditions to the file.
to_cif_file(
    aa,
    str(out_path),
    save_standard_annotations=True,
    chain_disambiguation="chain_iid",
)

# if you want to load again, you can do so like this:
reloaded = parse(
    str(out_path),
    config=ANNOTATED_CIF,  # this config ensures that all conditions are loaded from the file
)["assemblies"]["1"][0]

## 6. Emit many variants programmatically

Coordinate conditioning takes its value straight from the atom coordinates, so we can sweep a pinned atom and re-save. Here we push the `THR 87` tip atom (`OG1`) directly away from the ligand in 0.1 Å steps, writing one CIF per step — a batch that probes how far the pocket tolerates that contact. Point `generate.py` at the directory to run them all.

In [ ]:
sweep = np.where((aa.chain_id == "1.A") & (aa.res_id == 87) & (aa.atom_name == "OG1"))[0][0]
away = aa.coord[sweep] - aa.coord[aa.is_ligand].mean(axis=0)
away /= np.linalg.norm(away)  # unit vector pointing from the ligand toward the tip atom

variants_dir = OUT_DIR / "variants"
variants_dir.mkdir(exist_ok=True)
base_xyz = aa.coord[sweep].copy()
for i in range(10):
    variant = aa.copy()
    variant.coord[sweep] = base_xyz + away * 0.1 * (i + 1)
    to_cif_file(
        variant,
        str(variants_dir / f"variant_{i:02d}.cif"),
        save_standard_annotations=True,
        chain_disambiguation="chain_iid",
    )

## 7. Checking your work

If you load the saved annotated CIF into PyMOL with the PyMOL plugin (see the [annotated CIF guide](../guides/annotated_cif_inputs.md) for more information), you can visually inspect all annotations, bonds, and index groups to catch mistakes before running inference.

## 8. Visualize what you set

One interactive viewer over the annotated CIF, with a button per thing this notebook fixed.
Most buttons are AtomWorks selection queries (`aa.mask("...")`); the H-bond roles are read
through the condition object (`C_HB.mask`) since they are not part of the query syntax yet.

In [ ]:
import io, json, uuid
import biotite.structure.io.pdb as biopdb
from IPython.display import HTML

VIEWER_JS = "https://3Dmol.org/build/3Dmol-min.js"


def _to_pdb(sub):
    f = biopdb.PDBFile(); f.set_structure(sub)
    buf = io.StringIO(); f.write(buf); return buf.getvalue()


def selection_gallery(aa, selections, width=760, height=520):
    """Interactive viewer: one button per selection. Each entry is ``(label, query_or_mask)`` --
    a string is evaluated with ``aa.mask(query)``; a boolean array is used as given (for
    conditions the query syntax does not expose, such as H-bond roles)."""
    import string
    aa = aa.copy()                                  # 3Dmol via PDB needs single-char chain ids
    _uniq = list(dict.fromkeys(np.asarray(aa.chain_id).astype(str).tolist()))
    _cmap = {c: string.ascii_uppercase[i % 26] for i, c in enumerate(_uniq)}
    aa.chain_id = np.array([_cmap[c] for c in np.asarray(aa.chain_id).astype(str)])
    chain = np.asarray(aa.chain_id).astype(str)
    resi = aa.res_id.astype(int)
    aname = np.asarray(aa.atom_name).astype(str)
    items = []
    for label, sel in selections:
        if isinstance(sel, str):
            m = np.asarray(aa.mask(sel)); note = f'aa.mask("{sel}")'
        else:
            m = np.asarray(sel); note = 'C_HB.mask(aa)'
        atoms = [[str(chain[i]), int(resi[i]), str(aname[i])] for i in np.where(m)[0]]
        items.append({"label": label, "note": note, "n": int(m.sum()), "atoms": atoms})

    uid = "g" + uuid.uuid4().hex[:10]
    payload = ",\n".join(
        "{{label:{l},note:{q},n:{n},atoms:{a}}}".format(
            l=json.dumps(it["label"]), q=json.dumps(it["note"]),
            n=json.dumps(it["n"]), a=json.dumps(it["atoms"]))
        for it in items)
    buttons = "".join(
        f'<button id="{uid}_b{i}" onclick="{uid}_show({i})" '
        f'style="margin:2px 4px 2px 0;padding:4px 10px;border:1px solid #bbb;border-radius:4px;'
        f'background:#f4f4f4;cursor:pointer;font:12px monospace">{it["label"]}</button>'
        for i, it in enumerate(items))

    return HTML(f"""
<div style="font:13px/1.5 system-ui,sans-serif">
  <div style="margin-bottom:4px">{buttons}</div>
  <div id="{uid}_note" style="margin:4px 0 8px;color:#333;font:12px monospace">&nbsp;</div>
  <div id="{uid}" style="width:{width}px;height:{height}px;position:relative"></div>
  <div style="margin-top:6px;color:#555;font-size:12px">
    grey cartoon = structure &nbsp;|&nbsp; orange = ligand &nbsp;|&nbsp; green = selected atoms
  </div>
</div>
<script src="{VIEWER_JS}"></script>
<script>
(function() {{
  var data = [
{payload}
  ];
  var pdb = {json.dumps(_to_pdb(aa))};
  var cur = 0, viewer = null;
  function draw() {{
    var s = data[cur];
    viewer.clear();
    viewer.addModel(pdb, "pdb");
    viewer.setStyle({{}}, {{stick: {{colorscheme: "grayCarbon", radius: 0.10}}}});
    viewer.setStyle({{hetflag: true}}, {{stick: {{colorscheme: "orangeCarbon", radius: 0.22}}}});
    var ors = [];
    for (var k = 0; k < s.atoms.length; k++)
      ors.push({{chain: s.atoms[k][0], resi: s.atoms[k][1], atom: s.atoms[k][2]}});
    if (ors.length) {{
      viewer.addStyle({{or: ors}}, {{sphere: {{scale: 0.34, colorscheme: "greenCarbon"}}}});
      viewer.addStyle({{or: ors}}, {{stick: {{colorscheme: "greenCarbon", radius: 0.18}}}});
    }}
    viewer.zoomTo(); viewer.render();
    document.getElementById("{uid}_note").innerHTML = s.note + " &rarr; " + s.n + " atoms";
    for (var i = 0; i < data.length; i++) {{
      var b = document.getElementById("{uid}_b" + i);
      if (b) {{ b.style.background = (i === cur) ? "#cfe3cf" : "#f4f4f4";
                b.style.fontWeight = (i === cur) ? "bold" : "normal"; }}
    }}
  }}
  window["{uid}_show"] = function(i) {{ cur = i; draw(); }};
  function build() {{
    var el = document.getElementById("{uid}");
    if (!el || typeof $3Dmol === "undefined") {{ setTimeout(build, 120); return; }}
    viewer = $3Dmol.createViewer(el, {{backgroundColor: "white"}});
    draw();
  }}
  build();
}})();
</script>
""")

In [ ]:
check = parse(str(out_path), config=ANNOTATED_CIF)["assemblies"]["1"][0]
ensure_annotations(check, "is_ligand")
check = check[~np.isnan(check.coord).any(axis=1)]   # 3Dmol needs real coordinates

selection_gallery(check, [
    ("substrate",               "hetero"),
    ("coordinate-conditioned",  "mask_coordinate_1_atom"),
    ("sequence-fixed",          "mask_sequence_1_residue"),
    ("indexed residues",        "mask_index_1_residue"),
    ("H-bond donors/acceptors", np.asarray(C_HB.mask(check))),
])